Each target has an independent affine OLS fit across **all ratios**, separately within each seed. Panels (a–b) show 20 randomly selected target–seed observations and their fitted lines; panel (c) shows each experiment's in-sample R² ECDF across all its target–seed observations. Samples with undefined R² are omitted from the ECDF.

Expected existing file layout (root directory names can be changed):
```
RESULTS_ROOT/<config>/<dataset>/Seed=<seed>/N=<N>/ratio=<r>/shadow_<start>_<stop>.pkl
DATA_ROOT/<dataset>/Seed=<seed>/N=<N>/fpc_data_subsets.pkl
```
- Result chunks contain `target_stats`, `target_membership`, `X_eval_indices`, `M_shadow`, `start_idx`, and `stop_idx`. 
- Subset metadata contains `X_eval_indices` and `frame_sizes_by_ratio`. 
- Only load trusted pickle files. Dependencies: NumPy, Matplotlib, tueplots.

In [ ]:
from pathlib import Path
import pickle
import re
import numpy as np
from tueplots import bundles
import matplotlib.pyplot as plt
from cycler import cycler
from matplotlib.cm import get_cmap

cmap = get_cmap("tab10", 8)
palette = [cmap(i) for i in range(8)]
rc = bundles.iclr2024(usetex=False)
rc.update({
    "axes.prop_cycle": cycler(color=palette),
    "legend.frameon": False,
    "axes.grid": False,
})

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_ROOT = ROOT / "fpc_results"
DATA_ROOT = ROOT / "fpc_data"
OUTPUT_PATH = ROOT / "plots" / "variance_fits_two_configs.pdf"
SAVE_FIGURE = True

# Exactly two configs, one for each fit panel. Change seeds independently.
CONFIGS = [
    {"optimizer": "TABPFN", "dataset": "adult", "seeds": [42], "label": "Adult"},
    {"optimizer": "ViT", "dataset": "patch_camelyon", "seeds": [42], "label": "PatchCamelyon"},
]
N = 1000
RATIOS = [i / 10 for i in range(1, 10)]
CONDITION = "OUT"  # "IN" or "OUT"
N_PLOT_SAMPLES = 20
PLOT_SEED = 42

In [ ]:
class NumpyCompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        return super().find_class(module.replace("numpy._core", "numpy.core"), name)


def load_pickle(path):
    with path.open("rb") as stream:
        return NumpyCompatUnpickler(stream).load()


def load_variances(config, seed):
    relative = Path(config["dataset"]) / f"Seed={seed}" / f"N={N}"
    metadata = load_pickle(DATA_ROOT / relative / "fpc_data_subsets.pkl")
    target_ids = np.asarray(metadata["X_eval_indices"])
    observed, factors = [], []
    for ratio in RATIOS:
        folder = RESULTS_ROOT / config["optimizer"] / relative / f"ratio={ratio}"
        paths = sorted(p for p in folder.glob("shadow_*.pkl")
                       if re.fullmatch(r"shadow_\d+_\d+\.pkl", p.name))
        if not paths:
            raise FileNotFoundError(f"No shadow chunks in {folder}")
        first = load_pickle(paths[0])
        stats = np.full((first["M_shadow"], len(target_ids)), np.nan)
        membership = np.zeros(stats.shape, dtype=bool)
        for path in paths:
            chunk = first if path == paths[0] else load_pickle(path)
            np.testing.assert_array_equal(chunk["X_eval_indices"], target_ids)
            start, stop = chunk["start_idx"], chunk["stop_idx"]
            stats[start:stop] = chunk["target_stats"]
            membership[start:stop] = chunk["target_membership"]
        if not np.isfinite(stats).all():
            raise ValueError(f"Missing or nonfinite shadow statistics in {folder}")
        selected = membership if CONDITION == "IN" else ~membership
        if np.any(selected.sum(axis=0) < 2):
            raise ValueError(f"Too few {CONDITION} observations in {folder}")
        observed.append(np.array([
            stats[selected[:, j], j].var(ddof=1) for j in range(len(target_ids))
        ]))
        n_plus = metadata["frame_sizes_by_ratio"][ratio]
        factors.append(1 - (N - 1 if CONDITION == "IN" else N) / (n_plus - 1))
    return np.asarray(factors), np.stack(observed), target_ids


def fit_config(config):
    runs = []
    for seed in config["seeds"]:
        factors, observed, target_ids = load_variances(config, seed)
        design = np.column_stack([factors, np.ones_like(factors)])
        beta = np.linalg.lstsq(design, observed, rcond=None)[0]
        predicted = design @ beta
        ss_res = np.sum((observed - predicted) ** 2, axis=0)
        ss_tot = np.sum((observed - observed.mean(axis=0)) ** 2, axis=0)
        r2 = np.full(len(target_ids), np.nan)
        np.divide(ss_res, ss_tot, out=r2, where=ss_tot > 0)
        runs.append({"seed": seed, "target_ids": target_ids, "factors": factors,
                     "observed": observed, "beta": beta, "r2": 1 - r2})
    return runs

In [ ]:
assert len(CONFIGS) == 2, "Provide two experiment configs."
assert CONDITION in {"IN", "OUT"}
fit_results = [fit_config(config) for config in CONFIGS]
for config, runs in zip(CONFIGS, fit_results):
    values = np.concatenate([run["r2"] for run in runs])
    print(f"{config['label']} / {config['optimizer']}: {len(values)} target–seed observations, "
          f"median R²={np.nanmedian(values):.4f}")

In [ ]:
plot_rng = np.random.default_rng(PLOT_SEED)
f_grid = np.linspace(0, 1, 200)
sample_colors = plt.get_cmap("tab20")(np.linspace(0, 1, N_PLOT_SAMPLES))
selected_targets = []

with plt.rc_context(rc):
    fig, axes = plt.subplots(1, 3, figsize=(7, 3), layout="constrained")
    for panel, (config, runs) in enumerate(zip(CONFIGS, fit_results)):
        targets = [(run_index, j) for run_index, run in enumerate(runs)
                   for j in range(len(run["target_ids"]))]
        selection = plot_rng.choice(len(targets), size=N_PLOT_SAMPLES, replace=False)
        selected_targets.append([(runs[targets[i][0]]["seed"],
                                  runs[targets[i][0]]["target_ids"][targets[i][1]])
                                 for i in selection])
        ax = axes[panel]
        for index, color in zip(selection, sample_colors):
            run_index, j = targets[index]
            run = runs[run_index]
            ax.plot(f_grid, f_grid * run["beta"][0, j] + run["beta"][1, j],
                    color=color, lw=1.2, alpha=0.5)
            ax.plot(run["factors"], run["observed"][:, j], "o", color=color,
                    ms=4, markeredgewidth=0, alpha=0.35)
        ax.set(xlabel=r"$f$", ylabel=r"$\sigma^2$", xlim=(-0.02, 1.02),
               title=f"({chr(97 + panel)}) {config['label']}")
        values = np.concatenate([run["r2"] for run in runs])
        values = np.sort(values[np.isfinite(values)])
        cumulative = np.arange(1, len(values) + 1) / len(values)
        axes[2].step(values, cumulative, where="post", color=palette[panel],
                     lw=1.5, alpha=0.6, label=config["label"])
    axes[2].set(xlabel=r"$R_x^2$", ylabel="ECDF", ylim=(0, 1.02), title="(c)")
    axes[2].legend(fontsize=7)
    for ax in axes:
        ax.set_box_aspect(1.0)
    if SAVE_FIGURE:
        OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_PATH, bbox_inches="tight")
    plt.show()